In [ ]:
from google.colab import drive
drive.mount('/content/gdrive/')

Mounted at /content/gdrive/


In [ ]:
!cp "/content/gdrive/MyDrive/DATABASES/MSU_MFSD/face/msu_mfsd_faces.zip" .
!unzip -qq msu_mfsd_faces.zip

In [ ]:
import pandas as pd
import numpy as np
import glob
from pathlib import Path
import cv2
import matplotlib.pyplot as plt
import tqdm
import skimage.util as sk_noise
import PIL as pil

In [ ]:
import os
import shutil

In [ ]:
from sklearn.model_selection import train_test_split

## Select Images

In [ ]:
np.random.seed(42)

In [ ]:
"faces\<Data Type>\<Label>\<Scene>\<filename image>"
all_imgs_list = glob.glob('faces/**/**/**/**')
scene_list = glob.glob('faces/**/**/**')
len(scene_list), len(all_imgs_list)

(227, 63030)

In [ ]:
fixed_imgs_list = []
frame_number = 150
for i in scene_list:
    imgs_fold_list = glob.glob(f'{i}/**')
    if len(imgs_fold_list) < frame_number:
        frame_number = len(imgs_fold_list)
    else:
        frame_number = 150
    np.random.shuffle(imgs_fold_list)
    fixed_imgs_list += imgs_fold_list[:frame_number]
len(fixed_imgs_list)

34027

In [ ]:
df = pd.DataFrame([
    {
        'frame': i.split('/')[-1],
        'scene': i.split('/')[-2],
        'client': i.split('/')[-2].split('_')[1],
        'label': i.split('/')[-3],
        'data_type': i.split('/')[-4],
        'full_path': i

    } for i in fixed_imgs_list
])
print('Size DF:', df.shape[0])
df.sample(3)

Size DF: 34027


,frame,scene,client,label,data_type,full_path
33643,frame_56.jpg,real_client005_laptop_SD_scene01,client005,real,train,faces/train/real/real_client005_laptop_SD_scen...
28120,frame_221.jpg,attack_client003_android_SD_iphone_video_scene01,client003,attack,train,faces/train/attack/attack_client003_android_SD...
14971,frame_187.jpg,real_client049_laptop_SD_scene01,client049,real,test,faces/test/real/real_client049_laptop_SD_scene...


In [ ]:
df.to_csv('/content/gdrive/MyDrive/DATABASES/MSU_MFSD/face/msu_mfsd_faces_intradataset.csv', index=False)

In [ ]:
df_train = df[df.data_type=='train']
df_test = df[df.data_type=='test']
train, _ = train_test_split(df_train, stratify=df_train[['label', 'scene', 'client', 'data_type']], train_size = 1700/df_train.shape[0], random_state=42)
_, test = train_test_split(df_test, stratify=df_test[['label', 'scene', 'client', 'data_type']], test_size = 300/df_test.shape[0], random_state=42)
train.shape, test.shape, train.shape[0]+test.shape[0]

((1700, 6), (300, 6), 2000)

In [ ]:
df_for_iqa = pd.concat([train, test])
df_for_iqa.sample(3)

,frame,scene,client,label,data_type,full_path
32037,frame_63.jpg,real_client009_android_SD_scene01,client009,real,train,faces/train/real/real_client009_android_SD_sce...
27158,frame_164.jpg,attack_client034_laptop_SD_printed_photo_scene01,client034,attack,train,faces/train/attack/attack_client034_laptop_SD_...
19994,frame_28.jpg,attack_client008_android_SD_ipad_video_scene01,client008,attack,train,faces/train/attack/attack_client008_android_SD...


In [ ]:
!rm -r fas_iqa/
fas_iqa_dataset = []
for idx, row in df_for_iqa.iterrows():
    full_path = row['full_path']
    frame = row['frame']
    scene = row['scene']
    client = row['client']
    label = row['label']
    data_type = row['data_type']

    iqa_class = 'original'

    dir_path_new = f"fas_iqa/{data_type}/{iqa_class}/{label}/{scene}/"
    Path(f"{dir_path_new}").mkdir(parents=True, exist_ok=True)
    full_path_new = f"{dir_path_new}/{frame}"

    shutil.copy(full_path, full_path_new)
    if os.path.isfile(f"{full_path_new}"):
        fas_iqa_dataset.append({

            'frame': frame,
            'scene': scene,
            'client': client,
            'label': label,
            'data_type': data_type,
            'full_path': full_path_new,

            'iqa': 'original',
        })

df_fas_iqa = pd.DataFrame(fas_iqa_dataset)
df_fas_iqa.sample(3)

,frame,scene,client,label,data_type,full_path,iqa
1888,frame_296.jpg,attack_client032_android_SD_iphone_video_scene01,client032,attack,test,fas_iqa/test/original/attack/attack_client032_...,original
1587,frame_80.jpg,attack_client011_laptop_SD_printed_photo_scene01,client011,attack,train,fas_iqa/train/original/attack/attack_client011...,original
653,frame_196.jpg,attack_client055_android_SD_printed_photo_scene01,client055,attack,train,fas_iqa/train/original/attack/attack_client055...,original


In [ ]:
df_fas_iqa.to_csv('msu_iqa.csv', index=False)

In [ ]:
!zip -r "msu_iqa.zip" fas_iqa/ msu_iqa.csv -q

In [ ]:
!cp "msu_iqa.zip" "/content/gdrive/MyDrive/DATABASES/MSU_MFSD/face/"

In [ ]:
df_fas_iqa.shape

(2000, 7)

In [ ]:
scene_list = glob.glob('fas_iqa/**/**/**/**/**')
len(scene_list)

2000

END